# Phase 2 — Expert scenario study (protocol revision v2.0)

**Study:** *Where Does Institutional Constitution Fail? An Empirical Test of the Seven-Layer Energy Digital-Twin Model*

**Revision date:** 2026-10-02

## Why this revision exists
The original pre-data notebook specified a balanced incomplete-block design (CONTROL + 5 of 10 faults; 6 scenarios per participant). The implemented Google Forms instrument instead presents **CONTROL + all 10 fault scenarios to every eligible respondent**. This notebook freezes the implemented design before confirmatory outcome analysis.

The original notebook is preserved unchanged as `02_expert_scenario_study_legacy_bibd.ipynb` for auditability.

## Implemented design
- Complete within-participant repeated-measures design.
- Each eligible participant evaluates 11 scenarios: CONTROL + 10 faults.
- For each scenario, the participant answers institutional consequence, required system response, and primary formal responsibility.
- The primary institutional-consequence item has three response options in the live form: **Yes**, **No**, and **Not sure / insufficient information**.
- Confirmatory paired McNemar tests use only evaluable Yes/No pairs for each CONTROL-vs-fault contrast. `Not sure / insufficient information` is not recoded as Yes or No.
- Holm adjustment controls multiplicity across the 10 primary contrasts.
- Descriptive reporting includes the full three-category distribution, including uncertainty.
- Sensitivity analysis reports conclusions under alternative transparent handling of uncertain responses; no hidden recoding is permitted.

## Current field status at revision
Eight submissions had been received when this revision was made. Six passed the relevance screener and two were screened out. No confirmatory hypothesis test from these responses was used to choose or alter the analysis plan.


In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import numpy as np
import pandas as pd
from scipy.stats import binom, binomtest

PROTOCOL_VERSION = "2.0"
PROTOCOL_REVISION_DATE = "2026-10-02"
DESIGN = "complete_within_participant"

PHASE2_CONDITION_ORDER = [
    "CONTROL",
    "L2_MEAS_BIAS_DOWN",
    "L2_MEAS_BIAS_UP",
    "L2_MEAS_MISSING",
    "L3_TIMESTAMP",
    "L3_UNIT",
    "L3_ASSET_ID",
    "L4_WRONG_PROOF",
    "L4_MISSING_PROOF",
    "L5_RULE_ERROR",
    "L6_UNAUTHORIZED_ISSUE",
]
FAULT_CONDITIONS = PHASE2_CONDITION_ORDER[1:]

SCENARIOS_PER_ELIGIBLE_PARTICIPANT = 11
PRIMARY_RESPONSE_LEVELS = ["Yes", "No", "Not sure / insufficient information"]

assert len(PHASE2_CONDITION_ORDER) == 11
assert len(FAULT_CONDITIONS) == 10
assert SCENARIOS_PER_ELIGIBLE_PARTICIPANT == 11

print("Protocol", PROTOCOL_VERSION)
print("Design:", DESIGN)
print("Scenarios per eligible participant:", SCENARIOS_PER_ELIGIBLE_PARTICIPANT)


In [ ]:
# Prospective power calculation retained from the original design assumptions,
# but translated to the implemented complete repeated-measures design.
#
# Primary contrast: CONTROL vs each fault, paired exact McNemar.
# Final multiplicity control: Holm across 10 contrasts.
# Conservative sizing alpha: .05 / 10 = .005.
# Planning discordance: p01=.30, p10=.10 (net paired difference=.20).
# These are design assumptions, not estimates from the first 8 submissions.

FAMILYWISE_ALPHA = 0.05
N_PRIMARY_CONTRASTS = 10
POWER_ALPHA = FAMILYWISE_ALPHA / N_PRIMARY_CONTRASTS
TARGET_POWER = 0.80
P01 = 0.30
P10 = 0.10


# The original v1.1 prospective calculation already established that 135
# evaluable paired observations per fault achieve the planned >=80% power
# under p01=.30, p10=.10 and conservative alpha=.005. The design revision
# changes exposure, not these planning assumptions.
MIN_EVALUABLE_PAIRS = 135

# Every complete eligible expert supplies a pair for every fault, unless
# CONTROL or that fault is answered 'Not sure / insufficient information'.
# Therefore recruitment must exceed the evaluable-pair target.
# A conservative operational target of 160 complete eligible experts provides
# ~15.6% headroom over 135 pairs for uncertainty/incomplete evaluability.
OPERATIONAL_COMPLETE_ELIGIBLE_TARGET = 160

print("Conservative planning alpha:", POWER_ALPHA)
print("Minimum evaluable pairs under frozen planning assumptions:", MIN_EVALUABLE_PAIRS)
print("Operational complete-eligible recruitment target:", OPERATIONAL_COMPLETE_ELIGIBLE_TARGET)


In [ ]:
# Live Google Forms export validator.
# Set LIVE_CSV to the current export before running.

LIVE_CSV = Path(
    "data/Expert Validation of Energy Digital-Twin Certification Scenarios.csv"
)

PRIMARY_BASE = (
    "Under the GO/certification system, should this specific issue by itself "
    "require the certificate decision to be changed, blocked, or corrected?"
)
ACTION_BASE = "What should the certificate system do?"
RESP_BASE = (
    "Who should have the primary formal responsibility for resolving this issue "
    "under the certification system?"
)
ELIGIBILITY_COL = (
    "Are you currently or recently active in a professional or research area "
    "relevant to this study?"
)


def repeated_columns(columns, base, n=11):
    expected = [base] + [f"{base}.{i}" for i in range(1, n)]
    missing = [c for c in expected if c not in columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")
    return expected


def validate_live_export(path: Path):
    df = pd.read_csv(path)
    primary_cols = repeated_columns(df.columns, PRIMARY_BASE)
    action_cols = repeated_columns(df.columns, ACTION_BASE)
    resp_cols = repeated_columns(df.columns, RESP_BASE)

    eligible = df.loc[df[ELIGIBILITY_COL].eq("Yes")].copy()
    screened_out = df.loc[df[ELIGIBILITY_COL].eq("No")].copy()

    allowed_primary = set(PRIMARY_RESPONSE_LEVELS)
    observed_primary = set(pd.unique(eligible[primary_cols].values.ravel()))
    observed_primary = {x for x in observed_primary if pd.notna(x)}
    unexpected = observed_primary - allowed_primary
    if unexpected:
        raise ValueError(f"Unexpected primary responses: {sorted(unexpected)}")

    complete_all_11 = eligible[primary_cols + action_cols + resp_cols].notna().all(axis=1)

    return {
        "total_submissions": int(len(df)),
        "eligible": int(len(eligible)),
        "screened_out": int(len(screened_out)),
        "eligible_complete_all_11": int(complete_all_11.sum()),
        "observed_primary_levels": sorted(observed_primary),
    }

# Example:
# audit = validate_live_export(LIVE_CSV)
# print(audit)


In [ ]:
# Confirmatory analysis helpers.
# Expected long format columns:
# participant_id, condition_id, institutional_consequence


def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adjusted = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        value = min(1.0, (m - rank) * p[idx])
        running = max(running, value)
        adjusted[idx] = running
    return adjusted


def paired_mcnemar_complete_design(long_df: pd.DataFrame) -> pd.DataFrame:
    required = {"participant_id", "condition_id", "institutional_consequence"}
    missing = required - set(long_df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    wide = long_df.pivot(
        index="participant_id",
        columns="condition_id",
        values="institutional_consequence",
    )

    rows = []
    for fault in FAULT_CONDITIONS:
        pair = wide[["CONTROL", fault]].copy()
        # Confirmatory binary endpoint: only explicit Yes/No pairs are evaluable.
        pair = pair[
            pair["CONTROL"].isin(["Yes", "No"])
            & pair[fault].isin(["Yes", "No"])
        ]
        c = pair["CONTROL"].map({"Yes": 1, "No": 0})
        f = pair[fault].map({"Yes": 1, "No": 0})

        # Here Yes means intervention/change is required.
        n01 = int(((c == 0) & (f == 1)).sum())
        n10 = int(((c == 1) & (f == 0)).sum())
        discordant = n01 + n10
        pval = 1.0 if discordant == 0 else binomtest(
            min(n01, n10), discordant, 0.5, alternative="two-sided"
        ).pvalue
        rows.append({
            "fault_condition": fault,
            "evaluable_pairs": int(len(pair)),
            "control_no_fault_yes_n01": n01,
            "control_yes_fault_no_n10": n10,
            "raw_p": float(pval),
        })

    out = pd.DataFrame(rows)
    out["holm_p"] = holm_adjust(out["raw_p"].to_numpy())
    return out

# Full three-category distributions, including 'Not sure', must also be reported
# descriptively for every condition. Do not silently collapse uncertainty.


## Analysis freeze and reporting rules

1. Do not inspect significance patterns from the accumulating responses to change hypotheses, exclusions, or endpoint definitions.
2. Preserve the original BIBD notebook as a historical protocol version.
3. Treat the implemented all-11-scenario design as the confirmatory field design from protocol v2.0 onward.
4. Report the protocol deviation transparently in the manuscript and supplement.
5. Report `Not sure / insufficient information` explicitly. For the primary exact McNemar contrasts, only explicit Yes/No CONTROL–fault pairs are evaluable.
6. Use Holm correction across the 10 primary CONTROL–fault comparisons.
7. Report denominators for every contrast because uncertainty can reduce evaluable pairs.
8. Keep institutional adjudication (Phase 3) analytically independent of expert judgments until the prespecified synthesis stage.
9. Do not treat the eight submissions available at protocol revision as evidence for selecting the analysis plan.
